# Turnkey quickstart
Run a complete CPU-only evaluation, then read the audited four-file bundle.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path
import yaml

workspace = Path(tempfile.mkdtemp(prefix='turnkey-notebook-'))
config = yaml.safe_load(Path('configs/runs/smoke.yaml').read_text())
config['run']['out_dir'] = str(workspace / 'outputs')
config_path = workspace / 'smoke.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
completed = subprocess.run([sys.executable, '-m', 'turnkey.cli', 'run', '--config', str(config_path)], check=True, capture_output=True, text=True)
run_dir = Path(completed.stdout.strip().splitlines()[-1])
run_dir

In [ ]:
bundle = sorted(path.name for path in run_dir.iterdir() if path.is_file())
metrics = json.loads((run_dir / 'metrics.json').read_text())
audit = subprocess.run([sys.executable, '-m', 'turnkey.cli', 'audit', str(run_dir)], check=True, capture_output=True, text=True)
assert {'run.json', 'cases.jsonl', 'events.jsonl', 'metrics.json'} <= set(bundle)
assert audit.stdout.strip() == 'OK'
{'files': bundle, 'metric_sections': sorted(metrics), 'audit': 'passed'}